Optimierungsmodel:

Wir nehmen ein price-taker Model an, bei dem der (Ver-) Kaufspreis von außen vorgegeben wird. Ziel ist es, den Erlös

\sum_{i=0}^{n-1}  [ P_{entladen}(t_i) * \pi_{verkauf}(t_i) - P_{laden}(t_i) \pi_{kauf}(t_i) ] \Delta t

zu maximieren. Hierbei bezeichnet P_{(ent)laden}(t) die steuerbare Lade bzw. Entladeleistung und \pi_{verkauf} sowie \pi_{kauf} bezeichen den von außen vorgegebenen Marktpreis für Einspeisung und Verbrauch. Mit der binären Variable IST_LADEN(t_i)\in {0,1} bezeichnen wir den aktuellen Zustand des Systems. Wir verbieten gleichzeitiges Laden und Entladen durch die Vorschrift

0 < P_{entladen}(t_i) < [1-IST_LADEN(t_i)] * P_{max}
0 < P_{laden}(t_i) < IST_LADEN(t_i) * P_{max}.

Den Energieinhalt des Speichers bezeichnen wir mit E(t). Er folgt aus

E(t_{i+1}) = E(t_i) + [P_{laden}(t_i) - P_{entladen}(t_i)] * \Delta t.

Für E(t) nehmen wir die Einschränkung
E_{min} < E(t) < E_{max} vor.

Wir können das Problem in die Standardform eines gemischt-ganzzahligen linearen Programmes bringen. Dazu definieren wir

x_i      = P_{entladen}(t_i),
x_{i+n}  = P_{laden}(t_i),
x_{i+2n} = IST_LADEN(t_i),
x_{i+3n} = E(t_i)

und

c_i      = \pi_{verkauf}(t_i) \Delta t,     
c_{i+n}  = -\pi_{kauf}(t_i) \Delta t,
c_{i+2n} = 0
c_{i+3n} = 0.

Die Zielfunktion lautet damit ist c \cdot x.

Die Einschränkungen für Leistungsaufnahme/-abgabe können als

Ax <= b geschrieben werden, wobei

A_{i,i}    = 1,
A_{i,i+2n} = P_max,

A_{i+n, i+n}  = 1,
A_{i+n, i+2n} = - P_max,

b_i     = P_max,
b_{i+n} = 0

für i=0,1,...,n-1.

Die Veränderung des Energieinhaltes

E(t_0) = E0
E(t_{i+1}) - E(t_{i}) - P_{laden}(t_i) \Delta t + P_{entladen}(t_i) \Delta t = 0

schreiben wir als Nebenbedingung

A^{eq} x = b^{eq}

mit

A^{eq}_{3n,3n}  = 1,
b^{eq}_{3n}     =  e0

A^{eq}_{3n+j+1, 3n+j+1} = 1,
A^{eq}_{3n+j+1, 3n+j}   = -1,
A^{eq}_{3n+j+1, j}      = \Delta t,
A^{eq}_{3n+j+1, j+n}    = -\Delta t,

b^{eq}_{3n+1+j} = 0

für j=0,...,n-2.

Den Wertebereich der Variablen können wir explizit angeben.

Bemerkungen:
* Wir nehmen an, dass die Verluste beim Laden und Entladen symmetrisch sind.
* Für identischen Kauf- und Verkaufspreis brauchen wir keine binären Variablen ==> einfaches LP. ggf. hilfreich zum Testen.
* Energieinhalt am Ende relevant?

In [ ]:
import numpy as np

def return_problem(pi_verkauf, pi_kauf, p_max, e_min, e_max, e0, dt):

    # Anzahl Zeitschritte
    n = len(pi_kauf)

    # -------------------------------------------------
    # Variablen:
    # x = [P_discharge, P_charge, z, E]
    # -------------------------------------------------
    x = np.zeros(4 * n)

    # -------------------------------------------------
    # LP-Matrix (Ungleichungen)
    # -------------------------------------------------
    A = np.zeros((2 * n, 4 * n))
    b = np.zeros(2 * n)

    # Zielfunktion
    c = np.zeros(4 * n)

    # -------------------------------------------------
    # Gleichungen (Energie)
    # -------------------------------------------------
    A_eq = np.zeros((n, 4 * n))
    b_eq = np.zeros(n)

    # =================================================
    # Zielfunktion
    # =================================================
    for i in range(n):
        c[i] = pi_verkauf[i] * dt                 # P_entladen
        c[i + n] = -pi_kauf[i] * dt               # P_laden
        # z und E haben Kosten 0

    # =================================================
    # Ungleichungen: Lade-/Entladebeschränkung
    # =================================================
    for i in range(n):

        # P_discharge <= (1 - z) Pmax
        A[i, i] = 1
        A[i, i + 2 * n] = p_max
        b[i] = p_max

        # P_charge <= z Pmax
        A[i + n, i + n] = 1
        A[i + n, i + 2 * n] = -p_max
        b[i + n] = 0

    # =================================================
    # Energiegleichung
    # =================================================
    # E0 constraint
    A_eq[0, 3 * n] = 1
    b_eq[0] = e0

    # Dynamik
    for j in range(n - 1):

        # E_{j+1} - E_j
        A_eq[j + 1, 3 * n + (j + 1)] = 1
        A_eq[j + 1, 3 * n + j] = -1

        # + P_discharge * dt
        A_eq[j + 1, j] = dt

        # - P_charge * dt
        A_eq[j + 1, j + n] = -dt

        b_eq[j + 1] = 0

    # =================================================
    # Bounds (wichtig für MILP!)
    # =================================================
    bounds = []

    # P_discharge
    for i in range(n):
        bounds.append((0, p_max))

    # P_charge
    for i in range(n):
        bounds.append((0, p_max))

    # binary z
    for i in range(n):
        bounds.append((0, 1))

    # Energy
    for i in range(n):
        bounds.append((e_min, e_max))

    # =================================================
    # Ergebnis
    # =================================================
    return {
        "c": c,
        "A_ub": A,
        "b_ub": b,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
        "n": n
    }


    





    

